# FFT DMA demo

Feed 256 real samples into the `fft_wrapper` HLS core through the DMA `MM2S`
channel, read the 256-point complex FFT back through the `S2MM` channel, and
plot the magnitude spectrum.

> **Run this as root.** This board uses a custom non-XRT PYNQ backend
> (`zynq_cma_device`) that does MMIO and DMA over `/dev/mem`, which needs root.
> The FFT bitstream is already loaded at boot (it lives in `boot.bin`), so the
> PL is live and we only *attach* to it — we never reprogram it.

**Data path** (from `cfg/fft_demo_integ/bd_project.tcl`):

```
DDR --MM2S--> fft_wrapper.input_signal_stream
fft_wrapper.fft_output_stream --S2MM--> DDR
```

**Fixed-point formats** (from `cpp/fft_sysdef.h`):

| signal | C++ type | bits | scale (1.0 =) |
|--------|----------|------|----------------|
| input sample   | `ap_fixed<32,2>`  | 32 | `2**30` |
| FFT output re/im | `ap_fixed<32,9>` | 32 each | `2**23` |
| window coeff   | `ap_ufixed<18,0>` | 18 | `2**18` |

Each `S2MM` beat is 64 bits: **lower int32 = real, upper int32 = imag**.

In [ ]:
# On the Zybo Z7-10 (Zynq-7000, armv7) stock PYNQ 3.1 can't build a device
# (its layer needs XRT, which is aarch64-only). zynq_cma_device is a non-XRT
# shim shipped by meta-hls-pynq: importing it registers a /dev/mem-based Device
# as the active device, so Overlay/MMIO/allocate work. It MUST be imported
# before pynq, and the kernel needs root for /dev/mem -> run Jupyter as root.
import zynq_cma_device   # registers itself as the active PYNQ device

import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

# ---- design constants (keep in sync with cpp/fft_sysdef.h) ----
N        = 256          # FFT points
FS       = 48000        # codec sample rate [Hz] (Zybo SSM2603); adjust to yours

IN_FRAC  = 30           # ap_fixed<32,2>  -> 32-2 fractional bits
OUT_FRAC = 23           # ap_fixed<32,9>  -> 32-9 fractional bits
WIN_FRAC = 18           # ap_ufixed<18,0> -> 18 fractional bits

# ---- addresses (from bd_project.tcl) ----
# Staged next to this notebook by the gen-sdcard-image make target; the .hwh is
# named to match the .bit so Overlay() finds the hand-off.
BITSTREAM = 'fft_demo_top_wrapper.bit'

IN_SCALE  = float(1 << IN_FRAC)
OUT_SCALE = float(1 << OUT_FRAC)

## 1. Attach to the already-loaded overlay

The PL is already programmed, so we skip the bitstream download and only parse
the `.hwh` (hardware hand-off) to build the IP driver map — `download=False`
does exactly that without touching the running design. The `.bit`/`.hwh` pair
still needs to be present so PYNQ can find the metadata; point `BITSTREAM` at
them. The IP instance names below (`fft_dma`, `fft_wrapper_0`) come straight
from the block design in `bd_project.tcl`.

> With the `zynq_cma_device` shim active, this resolves through the non-XRT
> device: the `.hwh` is parsed directly (no `xclbinutil`), and IP registers are
> reached via `/dev/mem`.

## 1. Attach to the already-loaded overlay

The PL is already programmed, so we skip the bitstream download and only parse
the `.hwh` (hardware hand-off) to build the IP driver map — `download=False`
does exactly that without touching the running design. The `.bit`/`.hwh` pair
still needs to be present so PYNQ can find the metadata; point `BITSTREAM` at
them. The IP instance names below (`fft_dma`, `fft_wrapper_0`) come straight
from the block design in `bd_project.tcl`.

In [ ]:
AP_CTRL_OFFSET = 0x0
ol = Overlay(BITSTREAM, download=False)   # attach only; don't reprogram the PL

dma         = ol.fft_dma                # axi_dma
fft_wrapper = ol.fft_wrapper_0          # HLS core (AXI-Lite: window coeffs)

print(ol.ip_dict.keys())

## Enable IP
fft_wrapper.write(AP_CTRL_OFFSET, 0x81)
print(f"AP_CTRL_OFFSET = {fft_wrapper.read(AP_CTRL_OFFSET)}")

## 2. Program the window coefficients

`fft_wrapper` takes `window_coeffs[N/2]` (128 `ap_ufixed<18,0>` values) over its
AXI-Lite port. The array lives at some base offset inside the core's register
map. **Confirm `WIN_COEFF_OFFSET` against the generated driver header**
`xfft_wrapper_hw.h` (look for `XFFT_WRAPPER_..._WINDOW_COEFFS_BASE`); the value
below is the usual first-array offset but is design-dependent.

A Hann window is used here; write all `1.0` for a rectangular window.

In [ ]:
WIN_COEFF_OFFSET = 0x200   # <-- VERIFY against xfft_wrapper_hw.h

# Hann window over the full N points; the core mirrors it, so it only needs N/2.
hann = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N) / (N - 1))
coeffs = hann[: N // 2]

# ap_ufixed<18,0> saturates just below 1.0; clamp to the largest representable value.
max_q = (1 << WIN_FRAC) - 1
coeffs_q = np.clip(np.round(coeffs * (1 << WIN_FRAC)), 0, max_q).astype(np.uint32)

for i, c in enumerate(coeffs_q):
    fft_wrapper.write(WIN_COEFF_OFFSET + 4 * i, int(c))

print('wrote', len(coeffs_q), 'window coefficients')

## 3. Build a test input signal

Two tones so the spectrum is easy to read. Replace this with real captured audio
as needed — just keep it to `N` real samples in `[-2, 2)` (the `ap_fixed<32,2>`
range).

In [ ]:
n = np.arange(N)
f1, f2 = 3000.0, 9000.0     # Hz
sig = 0.6 * np.sin(2 * np.pi * f1 * n / FS) + 0.3 * np.sin(2 * np.pi * f2 * n / FS)

in_buf = allocate(shape=(N,), dtype=np.int32)
in_buf[:] = np.round(sig * IN_SCALE).astype(np.int32)

plt.figure(figsize=(9, 3))
plt.plot(n, sig)
plt.title('input signal (time domain)')
plt.xlabel('sample'); plt.ylabel('amplitude'); plt.grid(True)
plt.tight_layout(); plt.show()

## 4. Run the transfer

Output is 256 beats × 64 bits = 2048 bytes. Allocate it as `int32` of length
`2*N` so the interleaved [real, imag] pairs drop straight out. Arm the receive
channel first, then kick off the send.

In [ ]:
out_buf = allocate(shape=(2 * N,), dtype=np.int32)   # [re0, im0, re1, im1, ...]

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()

print('transfer complete')

## 5. Decode and plot the spectrum

Interpret the interleaved int32s as `ap_fixed<32,9>` (divide by `2**23`), form
the complex bins, and plot the one-sided magnitude. Bin `k` maps to
`k * FS / N` Hz.

In [ ]:
raw = np.array(out_buf).reshape(N, 2).astype(np.float64) / OUT_SCALE
spectrum = raw[:, 0] + 1j * raw[:, 1]     # bins 0..N-1 in natural order

half = N // 2
freqs = np.arange(half) * FS / N
mag   = np.abs(spectrum[:half])
mag_db = 20 * np.log10(mag + 1e-12)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.stem(freqs, mag)
ax1.set_ylabel('|X(f)|'); ax1.set_title('FFT magnitude (from DMA)'); ax1.grid(True)
ax2.plot(freqs, mag_db)
ax2.set_ylabel('dB'); ax2.set_xlabel('frequency [Hz]'); ax2.grid(True)
plt.tight_layout(); plt.show()

peak = freqs[np.argmax(mag)]
print(f'peak bin at {peak:.1f} Hz')

## 6. Sanity check against NumPy

Same input, same window, run through `numpy.fft` and overlay. They should line
up to within fixed-point quantisation.

In [ ]:
win_full = np.concatenate([coeffs_q / (1 << WIN_FRAC),
                           (coeffs_q / (1 << WIN_FRAC))[::-1]])
ref = np.fft.fft(sig * win_full)[:half]

plt.figure(figsize=(9, 3))
plt.plot(freqs, np.abs(ref), label='numpy')
plt.plot(freqs, mag, '--', label='hardware')
plt.legend(); plt.xlabel('frequency [Hz]'); plt.ylabel('magnitude')
plt.title('hardware vs numpy'); plt.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# free the contiguous buffers when done
in_buf.freebuffer()
out_buf.freebuffer()